05 — Export ONNX and upload to MinIO (OVMS layout)
==================================================
Goal: convert the trained PyTorch model to ONNX and store it where OpenVINO
Model Server expects it:

  s3://muffin-chihuahua/models/muffin-chihuahua/1/model.onnx

Later, in the Deploy model wizard, set Path = models/muffin-chihuahua
(the folder that contains version "1/", not the .onnx file itself).

Prerequisites:
  - Run 04 in the SAME kernel so `model` (and ideally `train_ds`) exist,
    OR this script will reload weights from model.pt.
  - Package `onnx` installed (see 01_install_deps.py).

In [ ]:
import os
from pathlib import Path

import boto3
import torch
import torch.nn as nn
from torchvision import models

# ---------------------------------------------------------------------------
# Ensure we have a model in memory (reload from checkpoint if needed)
# ---------------------------------------------------------------------------
DEVICE_EXPORT = "cpu"  # export on CPU for a portable ONNX graph
ckpt_path = Path("/opt/app-root/src/models/model.pt")

if "model" not in globals():
    print("No in-memory `model` — loading", ckpt_path)
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 2)
    model.load_state_dict(ckpt["model"])
    classes = ckpt.get("classes", ["chihuahua", "muffin"])
    print("Classes:", classes)
else:
    print("Using in-memory `model` from training cell")

model = model.to(DEVICE_EXPORT).eval()

# ---------------------------------------------------------------------------
# Local OVMS-style path: <name>/<version>/model.onnx
# ---------------------------------------------------------------------------
onnx_path = Path("/opt/app-root/src/models/muffin-chihuahua/1/model.onnx")
onnx_path.parent.mkdir(parents=True, exist_ok=True)

# Dummy batch matching serving input: NCHW 1×3×224×224
dummy = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    dummy,
    str(onnx_path),
    input_names=["input"],
    output_names=["output"],
    dynamo=False,  # classic exporter; fine for this lab
)
size_mb = round(onnx_path.stat().st_size / 1e6, 2)
print(f"ONNX written: {onnx_path} ({size_mb} MB)")

# ---------------------------------------------------------------------------
# Upload to MinIO
# ---------------------------------------------------------------------------
endpoint = os.environ.get(
    "AWS_S3_ENDPOINT",
    "http://minio.lab-minio.svc.cluster.local:9000",
)
bucket = os.environ.get("AWS_S3_BUCKET", "muffin-chihuahua")
s3_key = "models/muffin-chihuahua/1/model.onnx"

s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=os.environ.get("AWS_ACCESS_KEY_ID", "minio"),
    aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY", "minio123"),
)
s3.upload_file(str(onnx_path), bucket, s3_key)
print(f"Uploaded s3://{bucket}/{s3_key}")
print("\nStep 4 done — next: deploy the model (OVMS) with Path = models/muffin-chihuahua")